In [2]:
import torch
import torch.nn as nn

In [3]:
# Cross Correlation operation 
# Out = X, K
# X : feature input / channel input
# K : Convolution Kernel

def corr2d(X: torch.Tensor, K: torch.Tensor) -> torch.Tensor:
    h, w = K.shape
    Y = torch.zeros(
        (X.shape[0] - h + 1, X.shape[1] - w + 1)
    )
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (
                X[i: i + h, j: j + w] * K
            ).sum()
    return Y

In [4]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

In [5]:
# constructing a 2D conv layer

class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    # shapes should match
    def forward(self, x):
        return corr2d(x, self.weight) + self.bias


In [6]:
# construct an image pixel map 

X = torch.ones((6, 8))
X[:, 2:6] = 0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [17]:
# define a kernel 

# we want to detect and edge inside an image
# so we will use a simple logic for edge in 
# the above image
# if we get the absolute difference in the color :
#   we will get result other than 0
# otherwise: 
#   we will get the result == 0

K = torch.Tensor([(1.0, -1.0)])
K

tensor([[ 1., -1.]])

In [18]:
# Edge detection init
Y = corr2d(X, K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

In [22]:
# If we apply the same for transpose of the above image 
# We will see that now our kernel doesnt work because
# it detects only vertical edges

print("X (image) transpose :", X.t())

corr2d(X.t(), K)

X (image) transpose : tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])


tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

In [24]:
# To detect the edge in transposed image
# we need to transpose out kernel too in 
# order to iterate through the image 
# column wise diff

corr2d(X.t(), K.t())

tensor([[ 0.,  0.,  0.,  0.,  0.,  0.],
        [ 1.,  1.,  1.,  1.,  1.,  1.],
        [ 0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.],
        [-1., -1., -1., -1., -1., -1.],
        [ 0.,  0.,  0.,  0.,  0.,  0.]])

In [25]:
# Making the kernel learn through 

conv2d = nn.LazyConv2d(
    1, kernel_size=(1,2), bias = False
)

# The two-dimensional convolutional layer uses four-dimensional input and
# output in the format of (example, channel, height, width), where the batch
# size (number of examples in the batch) and the number of channels are both 1
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2 

for i in range(10):
    Y_hat = conv2d(X)
    # squared loss
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()

    # back prop
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i + 1}, loss {l.sum():.3f}')

epoch 2, loss 10.028
epoch 4, loss 3.344
epoch 6, loss 1.241
epoch 8, loss 0.487
epoch 10, loss 0.196


In [27]:
# Result very close to our edge detection

conv2d.weight.data.reshape(1, 2)

tensor([[ 0.9453, -1.0362]])

In [28]:
def compute_conv2d(conv2d, X):
    # (1, 1) = one sample and one channel (greyscale)
    X = X.reshape((1, 1) + X.shape)
    Y = conv2d(X)
    return Y.reshape(Y.shape[2:])


In [30]:
# Define a conv layer with padding
conv2d = nn.LazyConv2d(1, kernel_size=3, padding=1)
# random matrix
X = torch.rand(size=(8, 8))
compute_conv2d(conv2d, X).shape

torch.Size([8, 8])

In [ ]:
# We use a convolution kernel with height 5 and width 3. 
# The padding on either side of the height and width are 2 and 1, respectively
conv2d = nn.LazyConv2d(1, kernel_size=(5, 3), padding=(2, 1))
compute_conv2d(conv2d, X).shape

torch.Size([8, 8])

In [36]:
# Adding Stride
# skipping pixels while kernel calculations

conv2d = nn.LazyConv2d(1, kernel_size=3, padding=1, stride=2)
compute_conv2d(conv2d, X).shape

torch.Size([4, 4])

In [37]:
conv2d = nn.LazyConv2d(1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4))
compute_conv2d(conv2d, X).shape

torch.Size([2, 2])